# RNN, LSTM ও GRU

## এই notebook সম্পর্কে

এটি `example.py`-এর হুবহু কোড, notebook-এ বিভক্ত:

- Vanilla RNN ও LSTM-এর for-scratch NumPy forward pass
- vanishing-gradient সমস্যার একটি সরাসরি সংখ্যামূলক প্রদর্শন
- sequence দীর্ঘ হতে হতে, উভয় architecture-এ FIRST input-এ একটি ক্ষুদ্র পরিবর্তনের (nudge)
  প্রতি FINAL hidden state কতটা সাড়া দেয় তা মাপা হয়

**চালানো:** মূল ফোল্ডারে `python example.py`, অথবা এই notebook-এর cell-গুলো ক্রমান্বয়ে চালান।

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

INPUT_DIM = 4
HIDDEN_DIM = 8


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

## ১. Vanilla RNN forward pass

একটি RNN একবারে একটি timestep প্রক্রিয়া করে, প্রতিটি ধাপে একই weight matrix পুনর্ব্যবহার করে।
এখানে আমরা সম্পূর্ণ sequence পড়ে শুধু final hidden state ফেরত দিই।

In [ ]:
class VanillaRNN:
    def __init__(self, input_dim, hidden_dim):
        scale = 1.0 / np.sqrt(hidden_dim)
        self.Wxh = rng.normal(0, scale, size=(hidden_dim, input_dim))
        self.Whh = rng.normal(0, scale, size=(hidden_dim, hidden_dim))
        self.bh = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        h = np.zeros(self.hidden_dim)
        for x in x_seq:
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
        return h

## ২. LSTM forward pass

LSTM একটি অতিরিক্ত cell state রাখে, আর forget/input/output gate-গুলো সিদ্ধান্ত নেয়
কী মনে রাখবে, কী লিখবে, কী প্রকাশ করবে। cell state-এর আপডেট **additive**, তাই
gradient অনেক timestep পিছিয়ে প্রায় অপরিবর্তিত প্রবাহিত হতে পারে।

In [ ]:
class LSTM:
    def __init__(self, input_dim, hidden_dim, forget_bias=2.0):
        scale = 1.0 / np.sqrt(hidden_dim)
        z_dim = input_dim + hidden_dim
        self.Wf = rng.normal(0, scale, size=(hidden_dim, z_dim))
        self.Wi = rng.normal(0, scale, size=(hidden_dim, z_dim))
        self.Wg = rng.normal(0, scale, size=(hidden_dim, z_dim))
        self.Wo = rng.normal(0, scale, size=(hidden_dim, z_dim))
        # ধনাত্মক forget-gate bias একটি সুপরিচিত init কৌশল: এটি cell-টিকে
        # ডিফল্টভাবে "মনে রাখা" শুরু করতে দেয় (sigmoid(2.0) ~ 0.88) —
        # ডিফল্টভাবে ভুলে যাওয়ার (sigmoid(0.0) = 0.5) বদলে।
        self.bf = np.full(hidden_dim, forget_bias)
        self.bi = np.zeros(hidden_dim)
        self.bg = np.zeros(hidden_dim)
        self.bo = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        for x in x_seq:
            z = np.concatenate([h, x])
            f = sigmoid(self.Wf @ z + self.bf)
            i = sigmoid(self.Wi @ z + self.bi)
            g = np.tanh(self.Wg @ z + self.bg)
            o = sigmoid(self.Wo @ z + self.bo)
            c = f * c + i * g          # additive update -- vanilla RNN থেকে মূল পার্থক্য
            h = o * np.tanh(c)
        return h

## ৩. Gradient ক্ষয় পরিমাপ

Central-difference পদ্ধতিতে আমরা হিসাব করি: sequence-এর দৈর্ঘ্য বাড়লে, প্রথম input-এ একটি
ক্ষুদ্র পরিবর্তনের প্রতি final hidden state কতটা সংবেদনশীল থাকে। এই সংবেদনশীলতাই হলো
vanishing gradient-এর বাস্তব রূপ — প্রাথমিক token-এর training signal দূরের loss থেকে
শূন্যের দিকে সঙ্কুচিত হয়।

In [ ]:
def gradient_of_output_wrt_first_input(forward_fn, x_seq, eps=1e-4):
    """x_seq[0]-এর সাপেক্ষে sum(h_final)-এর central-difference gradient।"""
    x0 = x_seq[0]
    grad = np.zeros_like(x0)
    for k in range(len(x0)):
        seq_plus = [x.copy() for x in x_seq]
        seq_minus = [x.copy() for x in x_seq]
        seq_plus[0][k] += eps
        seq_minus[0][k] -= eps
        out_plus = forward_fn(seq_plus).sum()
        out_minus = forward_fn(seq_minus).sum()
        grad[k] = (out_plus - out_minus) / (2 * eps)
    return grad

## ৪. main(): ফলাফল

`main()` দুটি model-কে বাড়তে থাকা sequence দৈর্ঘ্যে (2 থেকে 40) পরীক্ষা করে এবং
RNN বনাম LSTM-এর gradient norm-এর তুলনা ছাপে।

In [ ]:
def main():
    print("Measuring: 'how much does the FIRST input still affect the FINAL")
    print("hidden state?' as the sequence gets longer -- this is exactly what")
    print("a vanishing gradient means in practice: the training signal for an")
    print("early token shrinks toward zero the further away the loss is.\n")

    rnn = VanillaRNN(INPUT_DIM, HIDDEN_DIM)
    lstm = LSTM(INPUT_DIM, HIDDEN_DIM, forget_bias=2.0)

    max_len = 40
    full_sequence = [rng.normal(size=INPUT_DIM) for _ in range(max_len)]

    seq_lengths = [2, 5, 10, 20, 30, 40]
    print(f"{'seq_len':>8}  {'RNN grad norm':>15}  {'LSTM grad norm':>16}")
    for T in seq_lengths:
        x_seq = full_sequence[:T]
        rnn_grad = gradient_of_output_wrt_first_input(rnn.forward, x_seq)
        lstm_grad = gradient_of_output_wrt_first_input(lstm.forward, x_seq)
        print(f"{T:>8}  {np.linalg.norm(rnn_grad):>15.8f}  {np.linalg.norm(lstm_grad):>16.8f}")

    print("\n-> The vanilla RNN's gradient norm collapses toward zero within a")
    print("   handful of steps: information from the first token is effectively")
    print("   erased by the time the network reaches the end of the sequence.")
    print("-> The LSTM's gradient decays far more slowly, because its cell state")
    print("   update (c_t = f_t*c_{t-1} + i_t*g_t) is additive rather than a")
    print("   repeated matrix multiplication -- exactly the mechanism described")
    print("   in the README. This is why LSTMs could learn much longer-range")
    print("   dependencies than vanilla RNNs.")

In [ ]:
main()